In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)

True

In [2]:

import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [3]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'

In [4]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Mohan. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4o-mini"

In [5]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I've noted your information. If you need to add more details or have any specific requests, feel free to let me know!

In [6]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I've recorded your information:

- **Name**: Mohan
  - **Role**: LLM engineer
  - **Activities**: Teaching a course about AI Agents and knowledgeable about the MCP protocol.
  
- **MCP Protocol**:
  - Connects agents with tools, resources, and prompt templates.
  - Facilitates integration of AI agents with various capabilities.

Let me know if you need any further assistance!

In [7]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Mohan . What do you know about me?")
    display(Markdown(result.final_output))

I know that you're Mohan, an LLM engineer. You are teaching a course about AI Agents, which includes the MCP protocol. If you'd like to share more or update any information, feel free to let me know!

In [8]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

In [9]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [10]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

As of July 23, 2026, Amazon's stock (AMZN) closed at **$233.66**, marking a **4.57% decline** in a single day. The shares saw a modest recovery in pre-market trading, rising to **$234.66**.

### Key Factors for Price Movement:
1. **Regulatory Scrutiny**: Amazon is under investigation by a U.S. Senate panel regarding potential undue Chinese influence on its marketplace. This has heightened investor concerns, further driven by ongoing antitrust lawsuits and allegations of deceptive business practices.
   
2. **Operational Changes**: The company has announced job cuts aimed at streamlining operations within its Artificial General Intelligence (AGI) division, focusing on enhancing projects that deliver direct customer value.

3. **Market Sentiment**: Despite the recent drop, analysts maintain a generally positive outlook. Major firms like Citi and KeyBanc recently reaffirmed their "Buy" ratings, with price targets ranging from **$325** to **$335**. The consensus average price target across analysts is around **$318.98**, indicating an anticipated upside of approximately **36.41%** from current levels.

4. **Earnings Anticipation**: The upcoming Q2 earnings report on July 30 is another focal point for investors. Analysts suggest that strong growth in Amazon's business segments, including a notable increase in gross sales on the Amazon Business platform, supports a continued bullish sentiment.

### Outlook:
In summary, while Amazon faces regulatory and operational challenges that contributed to its recent stock decline, analysts remain optimistic about the company's long-term growth potential, particularly in AWS and AI infrastructure. The consensus is a "Moderate Buy," with projections suggesting significant recovery potential.

### AGENTIC RAG INTEGRATION ####

In [11]:
vectordb_path = Path("memory/qdrant")
vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    vectorstore_tools = await server.list_tools()

vectorstore_tools

[Tool(name='qdrant-find', title=None, description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user', inputSchema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='qdrant-store', title=None, description='Keep the memory for later use, when you are asked to remember something.', inputSchema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'}, outpu

In [12]:
INSTRUCTIONS = """You research topics on the web and build up a knowledge base for later.
When you learn something worth keeping, store it in your knowledge base.
When you are asked what you know, search your knowledge base and answer from it."""

model = "gpt-3.5-turbo"

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as search_server:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
        agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[search_server, vector_server])
        with trace("research and store"):
            result = await Runner.run(agent, "Research the latest news on Nvidia and store the key facts in your knowledge base.", max_turns=20)
        display(Markdown(result.final_output))

I have researched the latest news on Nvidia and stored key facts in the knowledge base. Here are the key points:

1. Nvidia CEO Jensen Huang predicts Marvell as the next AI trillion-dollar giant, TCS opens an NVIDIA-powered AI engineering lab, and Yotta Data Services invests in Nvidia's latest chips for an AI hub.

2. Nvidia's DLSS 4.5 update with 6x frame generation is available, ePlane joins forces with Nvidia for aerospace innovation, and Uniphore raises $260 million in funding from Nvidia and others.

3. BlackBerry's QNX software safeguards Nvidia's self-driving platform, Arm's CEO introduces a disruptive new CPU, and IITian Rishabh Agarwal launches an Nvidia-backed startup.

4. Nvidia rewrites the AI storage rulebook at GTC 2026, partners with Microsoft for nuclear reactor deployment, and surpasses Tesla in global value creators ranking.

5. Jensen Huang emphasizes the profitability of AI and auctions his signature leather jacket for a high price.

In [13]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vector_server:
    agent = Agent(name="researcher", instructions=INSTRUCTIONS, model=model, mcp_servers=[vector_server])
    with trace("retrieve"):
        result = await Runner.run(agent, "Based on your knowledge base, what's the latest on Nvidia?")
    display(Markdown(result.final_output))

Here are some recent updates on Nvidia:

1. Nvidia has rewritten the AI storage rulebook at GTC 2026 and partnered with Microsoft to create a digital ecosystem for accelerating nuclear reactor deployment.
2. Nvidia has released the DLSS 4.5 update with 6x frame generation to boost gaming performance. They have also partnered with Indian eVTOL startup ePlane for aerospace innovation.
3. Nvidia CEO Jensen Huang predicts Marvell as the next AI trillion-dollar giant and TCS has opened an NVIDIA-powered AI engineering lab in Bengaluru. Hiranandani Group-backed Yotta Data Services will spend over $2 billion on Nvidia's latest chips for an artificial intelligence hub.
4. BlackBerry's QNX software will safeguard NVIDIA's self-driving platform. Arm's CEO claims the market needs his new CPU, which could disrupt the industry. IITian Rishabh Agarwal rejected Meta's offer to launch an NVIDIA-backed startup for AI scientists.
5. Nvidia CEO Jensen Huang mentioned that AI is 'insanely profitable' as he engages with billionaire families. Additionally, Jensen Huang's signature leather jacket is up for auction, expected to sell for a significant price.

### INTEGRATION ###

In [14]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        "args": ["--from", "git+https://github.com/massive-com/mcp_massive@v0.10.0", "mcp_massive"],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    market_params = {"command": "uv", "args": ["run", "-m", "backend.market_server"]}

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as server:
    market_tools = await server.list_tools()

market_tools

[Tool(name='lookup_share_price', title=None, description='This tool provides the current price of the given stock symbol.\n\nArgs:\n    symbol: the symbol of the stock\n', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'lookup_share_priceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None)]

In [15]:
instructions = "You answer questions about the stock market."
request = "What was the most recent price that Kawasaki Motosport traded at?"
model = "gpt-3.5-turbo"

async with MCPServerStdio(params=market_params, client_session_timeout_seconds=120) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The most recent price that Kawasaki Motosport traded at was $358.91.